<div align="center">

# Universidad de San Martín

## Infraestructura para Ciencia de Datos

### Licenciatura en Ciencia de Datos

<img src="../../logo.jpg" alt="Logo UNSAM" width="300"/>

---

</div>

# Ejercicio 06: El veredicto

---

## Objetivo
Practicar lo que hace alguien de MLOps cuando le toca decidir: **consultar el tracking y elegir**. Un equipo dejó **siete candidatos** registrados en MLflow, con sus métricas y nada más — ninguno viene con recomendación. Tu trabajo es mirarlos y decir **cuál iría a producción**, o que ninguno está listo.

> **Acá no se entrena nada.** Los candidatos ya están registrados: lo que practicás es **consultar MLflow** y leer lo que devuelve. El modelo de verdad lo entrenaste en el notebook de la clase, con los datos del pipeline.

> **Un solo archivo, autocontenido**: **Parte 1** deja los candidatos en el tracking y te muestra cómo consultarlos; **Parte 2** es tu veredicto. Al final, **📦 Entrega** genera tu `.txt`.

> **Nota:** usa el **MLflow del stack** (`localhost:5000`) y, si no responde, cae a una carpeta local. El `.txt` registra cuál se usó.

> Tu decisión **no se autocorrige**: no hay una respuesta que el script marque como buena. Lo que se lee es **por qué** elegiste lo que elegiste.

---

> **Sobre la entrega**: leé [`README.md`](README.md) en esta carpeta — el deliverable es **un `.txt` por estudiante** en `estudiantes/`, **no el notebook**.

## ⚠️ Antes de empezar: tu rama, con el material de esta clase

Para subir tu entrega tenés que estar parado en **tu rama personal** (la que creaste en la Clase 01: `estudiante/apellido-nombre`), con el material nuevo de `main` adentro. Si estás en `main` o `dev`, vas a romper el patrón de PR del curso.

**Recordá cómo se arma**: minúsculas, sin tildes y **un solo guión** separando apellido de nombre; los compuestos van pegados (`estudiante/garcialopez-mariajose`). Es la misma rama de la Clase 01: no crees una nueva.

```bash
# Ver en qué rama estás
git branch --show-current

# Al empezar CADA clase: traer lo nuevo de main a tu rama
git checkout main
git pull origin main
git checkout estudiante/apellido-nombre   # reemplazá por tu rama
git merge main --no-edit
```

> Si abriste esta notebook estando en `main`, no te cambies a tu rama sin el `merge`: la notebook todavía no está en tu rama. Cerrala, corré los cuatro comandos y volvé a abrirla.

> La sección **📦 Entrega** del final verifica esto programáticamente y te avisa si estás en una rama no esperada.

---
# Parte 1 — Los candidatos (una sola vez)

**¿Qué hacen estas celdas?**
1. Dejan en MLflow un experimento `clase06_veredicto` con **siete candidatos**, cada uno con sus params y sus métricas.
2. Los **consultan** de tres formas: todos juntos, filtrados y ordenados del lado del server, y en la UI.

**No tenés que tocar nada.** Solo correrlas en orden.

> Los candidatos son **casos de estudio**: un run de MLflow es una fila con params y métricas, y el modelo serializado es opcional — acá no hay `fit`, ni dataset, ni artefacto. Es el tablero que vas a mirar, no un entrenamiento.

In [ ]:
# ============================================================
# Los siete candidatos, cargados al tracking.
# ============================================================
# OJO: son CASOS DE ESTUDIO, no entrenamientos. Un run de MLflow es una fila
# con params y metricas -- el modelo serializado es opcional y aca no existe:
# no hay fit, no hay dataset, no hay artefacto. El modelo de verdad lo
# entrenaste vos mas arriba, con los datos del pipeline.
#
# La celda es idempotente: si los candidatos ya estan, no los vuelve a crear.
import mlflow
import requests

EXPERIMENTO = "clase06_veredicto"

# Mismo criterio que el resto del curso: el server del stack si esta, y si no,
# una carpeta local. El .txt de la entrega registra cual se uso.
try:
    requests.get("http://localhost:5000/health", timeout=5).raise_for_status()
    TRACKING, MODO = "http://localhost:5000", "server"
except Exception as e:
    TRACKING, MODO = "file:./mlruns_veredicto", "local"
    print(f"El server de MLflow no respondio ({type(e).__name__}): uso una carpeta local.")
    print("Si esperabas usar el del stack, revisa que este levantado: docker compose ps")
mlflow.set_tracking_uri(TRACKING)
mlflow.set_experiment(EXPERIMENTO)
print(f"Tracking: {MODO} ({TRACKING})")

CANDIDATOS = {
    #                        algoritmo             vent  n_tr n_te  tr    test  bal   f1    prec  rec
    "random_forest_7d":     ("RandomForest",        7,   240,  60, 0.71, 0.68, 0.67, 0.67, 0.66, 0.68),
    "random_forest_3d_deep": ("RandomForest",       3,   240,  60, 0.98, 0.54, 0.53, 0.52, 0.53, 0.51),
    "decision_tree_1d":     ("DecisionTree",        1,    12,   3, 0.88, 0.83, 0.82, 0.82, 0.81, 0.83),
    "logistic_3d":          ("LogisticRegression",  3,   240,  60, 0.67, 0.66, 0.53, 0.21, 0.85, 0.12),
    "gradient_boosting_7d": ("GradientBoosting",    7,   240,  60, 0.64, 0.61, 0.61, 0.60, 0.60, 0.61),
    "svm_3d":               ("SVM",                 3,   240,  60, 0.52, 0.51, 0.50, 0.49, 0.50, 0.49),
    "random_forest_7d_v2":  ("RandomForest",        7,   240,  60, 0.74, 0.69, 0.64, 0.62, 0.71, 0.55),
}

cliente = mlflow.MlflowClient()
exp_id = cliente.get_experiment_by_name(EXPERIMENTO).experiment_id
ya_estan = {r.info.run_name for r in cliente.search_runs([exp_id])}

for nombre_run, (algo, vent, n_tr, n_te, tr, te, bal, f1, prec, rec) in CANDIDATOS.items():
    if nombre_run in ya_estan:
        continue
    with mlflow.start_run(run_name=nombre_run):
        mlflow.set_tag("caso_de_estudio", "clase06_veredicto")
        mlflow.log_param("algoritmo", algo)
        mlflow.log_param("ventana_dias", vent)
        mlflow.log_param("n_fechas_train", n_tr)
        mlflow.log_param("n_fechas_test", n_te)
        mlflow.log_metric("accuracy_train", tr)
        mlflow.log_metric("accuracy_test", te)
        mlflow.log_metric("balanced_accuracy_test", bal)
        mlflow.log_metric("f1_test", f1)
        mlflow.log_metric("precision_test", prec)
        mlflow.log_metric("recall_test", rec)

nuevos = len(CANDIDATOS) - len(ya_estan & set(CANDIDATOS))
print(f"Candidatos en el experimento: {len(CANDIDATOS)}  (creados ahora: {nuevos})")

---
## Consultar el tracking

Tres formas de mirar lo mismo, y conviene tenerlas a mano:

- **`search_runs`** devuelve un DataFrame con params y métricas, una fila por run.
- El **mismo pedido filtrando y ordenando del lado del server** (`filter_string`, `order_by`): con cientos de runs no se bajan todos para mirarlos a mano.
- La **UI** (`localhost:5000`): seleccionar varios runs y compararlos, que es como se hace en el día a día.

In [ ]:
# ============================================================
# Consultar el tracking: esto es lo que se practica.
# ============================================================
import pandas as pd

COLUMNAS = ["tags.mlflow.runName", "params.algoritmo", "params.n_fechas_train",
            "params.n_fechas_test", "metrics.accuracy_train", "metrics.accuracy_test",
            "metrics.balanced_accuracy_test", "metrics.f1_test",
            "metrics.precision_test", "metrics.recall_test"]
CORTO = ["candidato", "algoritmo", "n_train", "n_test", "acc_train", "acc_test",
         "balanced", "f1", "precision", "recall"]

# 1) Todos los runs del experimento, como DataFrame.
runs = mlflow.search_runs(experiment_names=[EXPERIMENTO])
tabla = runs[COLUMNAS].copy()
tabla.columns = CORTO
print("Los siete candidatos:")
display(tabla.sort_values("candidato").reset_index(drop=True))

# 2) El mismo pedido, pero filtrando y ordenando DEL LADO DEL SERVER: con cientos
#    de runs no se bajan todos para mirarlos a mano.
mejores = mlflow.search_runs(
    experiment_names=[EXPERIMENTO],
    filter_string="metrics.accuracy_test > 0.6",
    order_by=["metrics.f1_test DESC"],
)
print("\nLos que pasan accuracy_test > 0.6, ordenados por f1_test:")
display(mejores[COLUMNAS].set_axis(CORTO, axis=1).reset_index(drop=True))

# 3) Y lo mismo se mira en la UI: http://localhost:5000 -> experimento
#    "clase06_veredicto" -> seleccionar varios runs -> "Compare".

---
# Parte 2 — Tu veredicto

Un solo ítem, con la misma anatomía que los ejercicios de las clases anteriores:

1. **Consigna** — qué tenés que devolver.
2. **🔗 En producción esto sería...** — para qué sirve en un equipo real.
3. **→ Es MLOps porque:** — por qué esto es parte del oficio y no del modelado.
4. **💡 Hint** — pista **conceptual**: qué mirar, no qué elegir.
5. **Resultado esperado** — qué queda registrado en tu `.txt`.

## V1. Elegí un candidato (o ninguno) y justificá

**Consigna:** mirá los siete candidatos de la Parte 1 y completá las tres variables de la celda de abajo: `candidato` (el nombre del run que promoverías a producción, o `'NINGUNO'`), `motivo` (por qué ese, en tus palabras) y `que_miraria_despues` (qué mirarías **en producción** para darte cuenta de que se degradó).

**🔗 En producción esto sería...** la reunión donde se decide qué modelo se publica. Nadie reentrena ahí: se mira el tablero, se discute contra qué se comparó y alguien firma. Lo que queda escrito es el **porqué**, que es lo que se revisa cuando el modelo falla tres meses después.

**→ Es MLOps porque:** no se trata de entrenar mejor, sino de **decidir con evidencia y dejar registro**. El tracking existe justamente para que esa decisión no dependa de la memoria de nadie.

**💡 Hint:** las métricas están en la tabla y en la UI. Cuatro cosas ayudan a leerlas:

| Métrica | Qué mira |
| :--- | :--- |
| `accuracy_train` vs `accuracy_test` | cuánto acierta sobre lo que ya vio contra datos nuevos. Si la primera vuela y la segunda no, memorizó en vez de aprender |
| `balanced_accuracy_test` | el acierto promedio **de cada clase por separado**: no se deja engañar por un modelo que casi siempre contesta lo mismo |
| `f1_test` (con `precision_test` y `recall_test`) | combina cuánto de lo que predijo era cierto (precisión) con cuánto de lo cierto encontró (recall). Se desploma cuando el modelo casi nunca se juega |
| `n_fechas_train` / `n_fechas_test` | sobre cuántos días se entrenó y se midió. Una métrica altísima medida sobre tres días no dice casi nada |

> Todas las métricas, salvo `accuracy_train`, están medidas sobre el **test**, y el test es **temporal**: los últimos días, nunca mezclados con el entrenamiento.

**Resultado esperado:** tu `.txt` con la tabla de los siete candidatos **leída de MLflow** (no la escribís vos), tu elección y tus dos textos. Si el candidato que ponés no existe entre los siete, queda marcado `CANDIDATO_INEXISTENTE`.

In [ ]:
# Tu veredicto. Completa las tres variables y despues corre la Entrega.
candidato = ''             # <-- el run que promoverias, o 'NINGUNO'
motivo = ''                # <-- por que ese
que_miraria_despues = ''   # <-- que mirarias en produccion para detectar que se degrado

---
## ✅ Llegaste al final

Miraste siete candidatos, los consultaste desde el tracking y tomaste una decisión con la evidencia a la vista. Eso es, literalmente, el trabajo: el modelo lo entrena cualquiera, **promoverlo es una decisión que alguien firma**.

Lo mismo pasa con el modelo real del pipeline: el DAG `crypto_ml` no elige nada — scorea el que esté marcado como `@champion` en el Registry. Cambiar ese alias **cambia lo que predice el pipeline sin tocar una línea de código**, y por eso la decisión pesa.

---

## 📦 Entrega

Generá tu archivo de entrega. **NO commitees el `.ipynb`** (es template compartido — generaría conflictos con el resto de los estudiantes). Reglas completas en [`README.md`](README.md).

> La entrega **relee los candidatos de MLflow** y escribe la tabla con lo que el tracking devuelve ahora: nada se autoreporta. Lo único que sale de vos son la elección y los dos textos.
>
> **Si te falta alguno de los textos, entregá igual**: queda con estado parcial.

In [ ]:
# Compuestos PEGADOS, con la mayuscula adentro: "Maria Jose" -> MariaJose, "Garcia Lopez" -> GarciaLopez.
# De ahi salen tu archivo (garcialopez-mariajose.txt) y tu rama (estudiante/garcialopez-mariajose):
# UN solo guion, el que separa apellido de nombre. Si los dejas separados, el script pega los espacios igual.
nombre = ''    # <-- Completar con tu nombre
apellido = ''  # <-- Completar con tu apellido
# Los usa la Entrega: el nombre de tu .txt y el codigo de verificacion.

In [ ]:
# ============================================================
# Entrega: las metricas salen de MLflow, no de lo que escribas aca.
# ============================================================
import hashlib
import re
import subprocess
import unicodedata
from datetime import date
from pathlib import Path

if not nombre.strip() or not apellido.strip():
    raise ValueError('Completa tu nombre y apellido en la celda anterior antes de ejecutar.')

# Se relee el experimento: lo que va al .txt es lo que el tracking devuelve ahora.
_runs = mlflow.search_runs(experiment_names=[EXPERIMENTO])
_t = _runs[COLUMNAS].set_axis(CORTO, axis=1).sort_values("candidato").reset_index(drop=True)
_nombres = set(_t["candidato"])

estado_eleccion = 'OK'
if not candidato.strip():
    estado_eleccion = 'SIN_ELECCION'
elif candidato.strip() != 'NINGUNO' and candidato.strip() not in _nombres:
    estado_eleccion = 'CANDIDATO_INEXISTENTE'
faltan_textos = [n for n, v in (('motivo', motivo), ('que_miraria_despues', que_miraria_despues))
                 if not v.strip()]

_filas = [
    f"  {r.candidato:<24} n_train={r.n_train:<4} n_test={r.n_test:<3} acc_train={r.acc_train:.2f} "
    f"acc_test={r.acc_test:.2f} balanced={r.balanced:.2f} f1={r.f1:.2f} "
    f"precision={r.precision:.2f} recall={r.recall:.2f}"
    for r in _t.itertuples()
]
_huella = ';'.join(f"{r.candidato}:{r.acc_test:.2f}:{r.f1:.2f}" for r in _t.itertuples())
codigo = hashlib.sha256(
    f"{apellido.strip().lower()}-{nombre.strip().lower()}-veredicto-{MODO}-{_huella}"
    f"-{candidato.strip()}-{date.today().isoformat()}".encode()).hexdigest()[:12].upper()

print('=' * 56)
print('        ENTREGA - CLASE 06 (El veredicto)')
print('=' * 56)
print(f'Estudiante: {nombre.strip()} {apellido.strip()}')
print(f'  Tracking: {MODO} ({TRACKING})')
print(f'  Candidatos leidos de MLflow: {len(_t)}')
print(f'  Eleccion: {candidato.strip() or "(vacia)"}  [{estado_eleccion}]')
if faltan_textos:
    print(f'  Faltan textos: {", ".join(faltan_textos)}  (podes entregar igual, con estado parcial)')
print(f'  Codigo: {codigo}')
print('=' * 56)


def slug(s):
    """Normaliza un nombre o apellido a una sola palabra.

    Los compuestos van PEGADOS: el guion se reserva como separador
    apellido-nombre, asi la rama nunca es ambigua.
        'Juan Pablo'   -> 'juanpablo'
        'Garcia Lopez' -> 'garcialopez'
    """
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    s = re.sub(r'[^a-zA-Z0-9]+', '', s).lower()   # sin espacios, guiones ni apostrofes
    return s


filename = f'{slug(apellido)}-{slug(nombre)}.txt'
try:
    _raiz = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'],
                                         stderr=subprocess.DEVNULL).decode().strip())
    target = _raiz / 'clase06' / 'ejercicios' / 'estudiantes' / filename
except Exception:
    target = Path('estudiantes') / filename   # fallback: relativo a esta notebook

_verbo = 'SOBRESCRIBIR (ya existe)' if target.exists() else 'crear'
print(f'Voy a {_verbo}: clase06/ejercicios/estudiantes/{filename}')
try:
    confirm = input('Confirmas? (s/n): ').strip().lower()
except Exception:
    print('(entorno sin input interactivo -- se confirma automaticamente)')
    confirm = 's'

if confirm in ('s', 'si', 'y', 'yes'):
    target.parent.mkdir(parents=True, exist_ok=True)
    contenido = (
        f'Apellido: {apellido.strip()}\n'
        f'Nombre: {nombre.strip()}\n'
        f'Tracking: {MODO} ({TRACKING})\n'
        f'Experimento: {EXPERIMENTO}\n'
        'Candidatos (leidos de MLflow, no autoreportados):\n'
        + '\n'.join(_filas) + '\n'
        f'Elegido: {candidato.strip() or "-"} [{estado_eleccion}]\n'
        f'Motivo: {motivo.strip() or "-"}\n'
        f'Que miraria despues: {que_miraria_despues.strip() or "-"}\n'
        f'Codigo: {codigo}\n'
        f'Fecha: {date.today().isoformat()}\n'
    )
    target.write_text(contenido, encoding='utf-8')
    print()
    print(f'Archivo creado: clase06/ejercicios/estudiantes/{filename}')
    print()
    try:
        rama_actual = subprocess.check_output(['git', 'branch', '--show-current'],
                                              stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        rama_actual = ''
    rama_ok = rama_actual.startswith('estudiante/')
    rama_push = rama_actual if rama_ok else 'estudiante/apellido-nombre'
    nota_rama = '' if rama_ok else '   <-- reemplaza por TU rama'
    print('Ahora subi SOLO ese archivo. Copia y pega estos tres comandos:')
    print()
    print(f'  git add clase06/ejercicios/estudiantes/{filename}')
    print('  git commit -m "clase06 (MachineLearning)"')
    print(f'  git push origin {rama_push}{nota_rama}')
else:
    print('No se escribio nada. Volve a correr esta celda cuando quieras confirmar.')

### 📦 Subí tu entrega

Subí **solo el `.txt`** (NO el `.ipynb`, que es template compartido).

**La celda de arriba te imprimió los tres comandos con tu archivo y tu rama ya puestos.** Copiálos de ahí y pegálos en la terminal, parado en la raíz del repo.

> **No uses `git add .`**: subirías también el `ejercicio.ipynb` modificado.

El `git push` actualiza tu PR abierto desde la Clase 01 — no abrís uno nuevo.

> Esta entrega es de **lectura y decisión**: no reemplaza al **TP Final**, que es el trabajo grande del cierre y se entrega la semana siguiente. Está en [`TpFinal/`](../TpFinal/).